## Etapa 1 - Extract (carregar e diagnosticar)
Carrego a base original direto da URL da plataforma, sem limpar nada ainda.
Objetivo: ver o estado bruto e descobrir como o pandas leu cada coluna.

In [1]:
import pandas as pd

url = "https://static.plataforma.grupoa.education/fmu/fmu_levelup/29c30e89-0e3e-4555-9649-ef46beb968eb.txt"

df = pd.read_csv(url, sep=",", encoding="utf-8")

df.head(10)

,Id,Nome,Data_Nascimento,Remedio_1,Remedio_2,Data_Inicio,Data_Fim,Glicemia_Inicial,Glicemia_Final,Dieta
0,P0001,Lucas Rocha Teixeira,01/12/1985,Não,Sim,30/07/2023,20/01/2024,781,14,Não
1,P0002,Leandro Ramos Mendes,11/07/1961,Sim,Não,15/06/2023,09/10/2023,174,127,Sim
2,P0003,Yasmin Ramos Rocha,10/11/1958,Sim,Não,13/12/2024,12/08/2025,212,-79,Não
3,P0004,Carlos Dias Ferreira,04/01/1990,Não,Sim,12/08/2024,02/06/2025,135,"108,5",Sim
4,p0005,Antônio Santos Andrade,12/08/1972,Sim,Não,05/07/2023,25/03/2024,5.7,82,Não
5,P0006,Vinícius Teixeira Ramos,20/09/1975,Não,Sim,13.06.2024,15/03/2025,211,190 mg/dL,Não
6,P0007,Maria Almeida Lima,20/05/1944,Não,Sim,07/12/2025,29/03/2026,152,128,Não
7,P0008,Camila Pereira Freitas,29/06/2000,Yes,Não,29-05-2024,10/05/2025,214,168,Sim
8,P0009,Vera Teixeira Cardoso,23/01/1976,1,Não,25/08/2024,10/03/2025,212,149,Sim
9,P0010,Larissa Araújo Ramos,16/04/1972,Não,Sim,16/06/2024,20/04/2025,174,146,Sim


In [2]:
# [E] Diagnóstico de entrada: formato e tipos lidos
print("Formato (linhas, colunas):", df.shape)
df.info()

Formato (linhas, colunas): (500, 10)
<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Id                500 non-null    str  
 1   Nome              500 non-null    str  
 2   Data_Nascimento   500 non-null    str  
 3   Remedio_1         500 non-null    str  
 4   Remedio_2         500 non-null    str  
 5   Data_Inicio       500 non-null    str  
 6   Data_Fim          500 non-null    str  
 7   Glicemia_Inicial  500 non-null    str  
 8   Glicemia_Final    500 non-null    str  
 9   Dieta             500 non-null    str  
dtypes: str(10)
memory usage: 39.2 KB


In [3]:
# [E] Quantos valores nulos já vieram por coluna
df.isna().sum()

Id                  0
Nome                0
Data_Nascimento     0
Remedio_1           0
Remedio_2           0
Data_Inicio         0
Data_Fim            0
Glicemia_Inicial    0
Glicemia_Final      0
Dieta               0
dtype: int64

In [4]:
# [D] Diagnóstico das colunas de categoria (bandeiras sim/não)
for col in ["Remedio_1", "Remedio_2", "Dieta"]:
    print(f"--- {col} ---")
    print(df[col].value_counts(dropna=False))
    print()

--- Remedio_1 ---
Remedio_1
Sim     227
Não     183
NÃO       9
não       9
s         8
N         8
sim       8
Yes       7
nao       7
S         7
Sim       7
1         6
Não       5
0         3
SIM       3
n         3
Name: count, dtype: int64

--- Remedio_2 ---
Remedio_2
Não    237
Sim    225
nao      7
S        6
Yes      5
NÃO      4
0        4
não      4
N        3
1        2
sim      2
SIM      1
Name: count, dtype: int64

--- Dieta ---
Dieta
Sim     216
Não     206
N         9
não       9
Sim       9
Yes       8
SIM       8
NÃO       7
0         7
Não       6
1         4
nao       4
sim       4
S         3
Name: count, dtype: int64



In [5]:
# [D] Diagnóstico 2: normalizar a superfície (tirar espaço + minúsculo) e re-contar
for col in ["Remedio_1", "Remedio_2", "Dieta"]:
    print(f"--- {col} ---")
    print(df[col].str.strip().str.lower().value_counts(dropna=False))
    print()

--- Remedio_1 ---
Remedio_1
sim    245
não    206
s       15
n       11
yes      7
nao      7
1        6
0        3
Name: count, dtype: int64

--- Remedio_2 ---
Remedio_2
não    245
sim    228
nao      7
s        6
yes      5
0        4
n        3
1        2
Name: count, dtype: int64

--- Dieta ---
Dieta
sim    237
não    228
n        9
yes      8
0        7
1        4
nao      4
s        3
Name: count, dtype: int64



In [6]:
# [T-a] Padronizar Remedio_1, Remedio_2 e Dieta para Sim/Não
mapa_sim_nao = {
    "sim": "Sim", "s": "Sim", "yes": "Sim", "1": "Sim",
    "não": "Não", "nao": "Não", "n": "Não", "0": "Não",
}

for col in ["Remedio_1", "Remedio_2", "Dieta"]:
    normalizado = df[col].str.strip().str.lower()   # mesma normalização do diagnóstico
    df[col] = normalizado.map(mapa_sim_nao)          # token -> canônico; o que não estiver no mapa vira NaN

# [V] Conferir: sobrou algum valor que o mapa não reconheceu?
df[["Remedio_1", "Remedio_2", "Dieta"]].isna().sum()

Remedio_1    0
Remedio_2    0
Dieta        0
dtype: int64

In [7]:
# [D] Diagnóstico da lógica entre colunas: combinação Remedio_1 x Remedio_2
pd.crosstab(df["Remedio_1"], df["Remedio_2"])

Remedio_2,Não,Sim
Remedio_1,,
Não,12,215
Sim,247,26


In [8]:
# [T-b] Remover pacientes com combinação inválida de remédios (ambos / nenhum)
valido = (df["Remedio_1"] == "Sim") ^ (df["Remedio_2"] == "Sim")   # XOR: exatamente um "Sim"
print("Linhas antes:", len(df))
print("Removidas:", (~valido).sum())
df = df[valido].copy()
print("Linhas depois:", len(df))

Linhas antes: 500
Removidas: 38
Linhas depois: 462


In [9]:
# [T-c] Criar coluna única "Medicamento" (qual dos dois o paciente tomou)
import numpy as np
df["Medicamento"] = np.where(df["Remedio_1"] == "Sim", "Remedio_1", "Remedio_2")

df["Medicamento"].value_counts()

Medicamento
Remedio_1    247
Remedio_2    215
Name: count, dtype: int64

In [10]:
# [D] Diagnóstico da glicemia, camada 1: quais valores NÃO são número puro?
for col in ["Glicemia_Inicial", "Glicemia_Final"]:
    convertido = pd.to_numeric(df[col], errors="coerce")   # o que não dá pra virar número vira NaN
    nao_numericos = df.loc[convertido.isna(), col]          # pega o texto ORIGINAL dos que falharam
    print(f"--- {col}: {len(nao_numericos)} valores não-numéricos ---")
    print(nao_numericos.value_counts())
    print()

--- Glicemia_Inicial: 29 valores não-numéricos ---
Glicemia_Inicial
112 mg/dL    2
231,1        1
153 mg/dL    1
100 mg/dL    1
174,6        1
132 mg/dL    1
147 mg/dL    1
108,1        1
165,8        1
209,7        1
264 mg/dL    1
181,2        1
193 mg/dL    1
169,2        1
115,2        1
177 mg/dL    1
214,9        1
170,3        1
173 mg/dL    1
192,4        1
125,8        1
143 mg/dL    1
172 mg/dL    1
179,5        1
128,6        1
144,3        1
104 mg/dL    1
190,2        1
Name: count, dtype: int64

--- Glicemia_Final: 32 valores não-numéricos ---
Glicemia_Final
180,4        2
108,5        1
190 mg/dL    1
92 mg/dL     1
77 mg/dL     1
87 mg/dL     1
107,5        1
95 mg/dL     1
159,8        1
132 mg/dL    1
173 mg/dL    1
96 mg/dL     1
110,7        1
131,7        1
165 mg/dL    1
157,9        1
136 mg/dL    1
162 mg/dL    1
77,5         1
105,2        1
230,6        1
199,1        1
150,8        1
133 mg/dL    1
80,6         1
93 mg/dL     1
153,7        1
138,1        1
1

In [11]:
# [T-d] Glicemia camada 1 (formato): recuperar texto e converter para número
for col in ["Glicemia_Inicial", "Glicemia_Final"]:
    serie = (df[col]
             .str.replace("mg/dL", "", regex=False)   # remove a unidade grudada
             .str.replace(",", ".", regex=False)       # vírgula decimal -> ponto
             .str.strip())                             # limpa espaços que sobraram
    df[col] = pd.to_numeric(serie, errors="coerce")    # agora vira número de verdade

# [V] Conferência: sobrou algo não convertido? E qual a cara dos números agora?
print(df[["Glicemia_Inicial", "Glicemia_Final"]].isna().sum())
print(df[["Glicemia_Inicial", "Glicemia_Final"]].describe())

Glicemia_Inicial    0
Glicemia_Final      0
dtype: int64
       Glicemia_Inicial  Glicemia_Final
count        462.000000      462.000000
mean         165.669048      136.559091
std          115.264238      113.223237
min         -140.000000     -138.000000
25%          137.000000      101.000000
50%          165.000000      131.000000
75%          190.150000      159.950000
max         1012.000000     1328.000000


In [12]:
# [D] Glicemia camada 2 (plausibilidade): olhar os extremos de cada coluna
for col in ["Glicemia_Inicial", "Glicemia_Final"]:
    ordenado = df[col].sort_values()
    print(f"--- {col} ---")
    print("10 menores:", ordenado.head(10).tolist())
    print("10 maiores:", ordenado.tail(10).tolist())
    print()

--- Glicemia_Inicial ---
10 menores: [-140.0, -134.0, -132.0, -132.0, -130.0, -126.0, -122.0, -103.0, -80.0, -70.0]
10 maiores: [273.0, 590.0, 593.0, 781.0, 793.0, 874.0, 930.0, 944.0, 1000.0, 1012.0]

--- Glicemia_Final ---
10 menores: [-138.0, -132.0, -114.0, -107.0, -103.0, -81.0, -79.0, 4.1, 4.1, 4.3]
10 maiores: [242.0, 557.0, 564.0, 636.0, 651.0, 653.0, 701.0, 742.0, 1279.0, 1328.0]



In [13]:
# [T-e] Glicemia camada 2 (plausibilidade): invalidar valores fora da faixa fisiológica
PISO = 20     # <-- Piso minimo para uma glicemia lógica em pessoas
TETO = 500    # <-- Olhando para a saida anterior, a um pulo brusco de valor; 273 > 590 e 242 > 557.
for col in ["Glicemia_Inicial", "Glicemia_Final"]:
    fora = (df[col] < PISO) | (df[col] > TETO)   # marca o que está fora da faixa
    print(f"{col}: {int(fora.sum())} valores invalidados")
    df.loc[fora, col] = np.nan                    # troca os inválidos por NaN

# conta quantas LINHAS ficaram com glicemia faltando em qualquer uma das duas colunas
linhas_afetadas = df[["Glicemia_Inicial", "Glicemia_Final"]].isna().any(axis=1).sum()
print("Linhas afetadas (serão removidas no próximo passo):", int(linhas_afetadas))

Glicemia_Inicial: 37 valores invalidados
Glicemia_Final: 35 valores invalidados
Linhas afetadas (serão removidas no próximo passo): 69


In [14]:
# [D] Antes de remover: a glicemia inválida está concentrada em algum grupo?
invalida = df[["Glicemia_Inicial", "Glicemia_Final"]].isna().any(axis=1)
print("Por medicamento:")
print(pd.crosstab(df["Medicamento"], invalida, normalize="index"))
print("\nPor dieta:")
print(pd.crosstab(df["Dieta"], invalida, normalize="index"))

Por medicamento:
col_0           False     True 
Medicamento                    
Remedio_1    0.846154  0.153846
Remedio_2    0.855814  0.144186

Por dieta:
col_0     False     True 
Dieta                    
Não    0.836910  0.163090
Sim    0.864629  0.135371


In [15]:
# [T-f] Remover as linhas com glicemia inválida (fora da faixa fisiológica)
print("Linhas antes:", len(df))
df = df.dropna(subset=["Glicemia_Inicial", "Glicemia_Final"]).copy()
print("Linhas depois:", len(df))

Linhas antes: 462
Linhas depois: 393


In [16]:
# [D] Diagnóstico das datas: quais valores NÃO parseiam como data?
for col in ["Data_Nascimento", "Data_Inicio", "Data_Fim"]:
    parsed = pd.to_datetime(df[col], errors="coerce", dayfirst=True)
    falhas = df.loc[parsed.isna(), col]
    print(f"--- {col}: {int(parsed.isna().sum())} não parsearam ---")
    print(falhas.value_counts().head(20))
    print()

--- Data_Nascimento: 45 não parsearam ---
Data_Nascimento
26 de dezembro de 1982    1
22/08/84                  1
05-10-1992                1
17/09/57                  1
24-06-1959                1
1990-06-04                1
31.01.1959                1
25 de março de 1948       1
27.05.1999                1
1995-10-22                1
30 de janeiro de 1961     1
14-12-1977                1
1 de julho de 1982        1
01.04.1981                1
4 de novembro de 1948     1
10/29/1963                1
16-11-1955                1
1965-09-03                1
27/04/46                  1
30-10-1982                1
Name: count, dtype: int64

--- Data_Inicio: 58 não parsearam ---
Data_Inicio
13.06.2024                1
29-05-2024                1
21/07/23                  1
26-09-2023                1
27.02.2024                1
2025-01-19                1
12-11-2023                1
2025-06-06                1
26 de março de 2025       1
2024-03-31                1
12/15/2023               

In [17]:
# [D2] Recuperação de datas, passo 1: ler cada valor no seu próprio formato
for col in ["Data_Nascimento", "Data_Inicio", "Data_Fim"]:
    parsed = pd.to_datetime(df[col], errors="coerce", dayfirst=True, format="mixed")
    restantes = df.loc[parsed.isna(), col]
    print(f"--- {col}: {int(parsed.isna().sum())} ainda falham ---")
    print(restantes.value_counts().head(20))
    print()

--- Data_Nascimento: 12 ainda falham ---
Data_Nascimento
26 de dezembro de 1982     1
25 de março de 1948        1
30 de janeiro de 1961      1
1 de julho de 1982         1
4 de novembro de 1948      1
5 de setembro de 1960      1
12 de julho de 1976        1
3 de julho de 1980         1
11 de julho de 1990        1
23 de junho de 1965        1
1 de janeiro de 1982       1
15 de fevereiro de 1942    1
Name: count, dtype: int64

--- Data_Inicio: 12 ainda falham ---
Data_Inicio
26 de março de 2025       1
12 de dezembro de 2023    1
22 de abril de 2023       1
18 de março de 2025       1
10 de agosto de 2024      1
4 de outubro de 2025      1
28 de março de 2023       1
23 de julho de 2023       1
9 de maio de 2023         1
6 de setembro de 2025     1
4 de dezembro de 2023     1
14 de abril de 2023       1
Name: count, dtype: int64

--- Data_Fim: 7 ainda falham ---
Data_Fim
19 de fevereiro de 2025    1
2 de julho de 2025         1
2 de agosto de 2025        1
19 de outubro de 2026      

In [18]:
# [T-g] Datas camada 1 (formato): converter mês por extenso (português) e parsear tudo
meses_pt = {
    "janeiro": "01", "fevereiro": "02", "março": "03", "abril": "04",
    "maio": "05", "junho": "06", "julho": "07", "agosto": "08",
    "setembro": "09", "outubro": "10", "novembro": "11", "dezembro": "12",
}

for col in ["Data_Nascimento", "Data_Inicio", "Data_Fim"]:
    texto = df[col].str.lower().str.strip()                 # normaliza caixa e espaços
    texto = texto.str.replace(" de ", "/", regex=False)     # "26 de dezembro de 1982" -> "26/dezembro/1982"
    for nome, numero in meses_pt.items():
        texto = texto.str.replace(nome, numero, regex=False)  # "dezembro" -> "12"
    df[col] = pd.to_datetime(texto, errors="coerce", dayfirst=True, format="mixed")

# [V] Quantas datas ainda ficaram sem parsear (NaT) em cada coluna?
print(df[["Data_Nascimento", "Data_Inicio", "Data_Fim"]].isna().sum())

Data_Nascimento    0
Data_Inicio        0
Data_Fim           0
dtype: int64


In [19]:
# [D] Diagnóstico lógico das datas
hoje = pd.Timestamp.today()

nasc_futuro = df["Data_Nascimento"] > hoje                      # nascer no futuro
nasc_depois = df["Data_Nascimento"] > df["Data_Inicio"]         # nascer depois de ser tratado
fim_antes   = df["Data_Fim"] < df["Data_Inicio"]                # tratamento acaba antes de começar

print("Nascimento no futuro:", int(nasc_futuro.sum()))
print("Nascimento depois do início do tratamento:", int(nasc_depois.sum()))
print("Data_Fim antes da Data_Inicio:", int(fim_antes.sum()))

Nascimento no futuro: 10
Nascimento depois do início do tratamento: 10
Data_Fim antes da Data_Inicio: 25


In [20]:
# [D] Olhar os 25 pares com Data_Fim antes da Data_Inicio
fim_antes = df["Data_Fim"] < df["Data_Inicio"]
print(df.loc[fim_antes, ["Data_Inicio", "Data_Fim"]].head(25))

    Data_Inicio   Data_Fim
18   2024-11-19 2024-05-21
31   2024-08-16 2024-04-22
36   2024-10-12 2024-04-26
68   2023-11-12 2023-10-24
76   2024-03-31 2023-12-24
84   2023-02-23 2023-01-22
165  2025-08-14 2025-06-10
167  2023-04-22 2022-11-06
171  2023-09-17 2023-05-08
172  2025-11-01 2025-05-12
178  2025-12-18 2025-10-15
214  2025-03-28 2024-11-27
221  2023-12-16 2023-10-23
281  2024-11-24 2024-07-02
287  2024-12-20 2024-10-07
295  2024-01-15 2023-07-02
302  2024-09-23 2024-05-03
339  2025-09-06 2025-04-11
343  2025-11-09 2025-06-27
361  2023-04-09 2023-03-07
435  2025-05-09 2025-01-22
458  2023-04-17 2023-02-15
474  2024-04-15 2024-02-22
481  2024-10-22 2024-06-24
490  2024-06-11 2024-02-10


In [21]:
# [T-h] Corrigir pares de data invertidos (Inicio/Fim trocados de coluna)
fim_antes = df["Data_Fim"] < df["Data_Inicio"]
df.loc[fim_antes, ["Data_Inicio", "Data_Fim"]] = df.loc[fim_antes, ["Data_Fim", "Data_Inicio"]].values

# [V] Conferir: ainda existe fim antes do início?
print("Pares incoerentes restantes:", int((df["Data_Fim"] < df["Data_Inicio"]).sum()))

Pares incoerentes restantes: 0


In [22]:
# [D] Olhar as datas de nascimento impossíveis (nasceu depois de já estar em tratamento)
nasc_ruim = df["Data_Nascimento"] > df["Data_Inicio"]
print(df.loc[nasc_ruim, ["Data_Nascimento", "Data_Inicio"]])

    Data_Nascimento Data_Inicio
32       2057-09-17  2023-08-19
153      2029-05-20  2025-12-19
182      2046-04-27  2023-11-17
196      2028-04-13  2025-03-18
272      2030-12-27  2024-09-28
275      2059-05-02  2023-04-06
331      2028-08-06  2023-01-24
341      2028-07-05  2024-08-06
444      2030-03-23  2025-02-06
470      2056-10-01  2023-01-17


In [23]:
# [T-i] Invalidar datas de nascimento impossíveis (sem evidência para recuperar o valor real)
nasc_ruim = df["Data_Nascimento"] > df["Data_Inicio"]
print("Nascimentos impossíveis invalidados:", int(nasc_ruim.sum()))
df.loc[nasc_ruim, "Data_Nascimento"] = pd.NaT

# [V] Conferir: nascimento com 10 ausentes e nenhuma linha perdida
print("NaT em Data_Nascimento:", int(df["Data_Nascimento"].isna().sum()))
print("Total de linhas:", len(df))

Nascimentos impossíveis invalidados: 10
NaT em Data_Nascimento: 10
Total de linhas: 393


In [25]:
# [T-j] Padronizar Id para o formato P0000
numeros = df["Id"].str.extract(r"(\d+)")[0]     # extrai só os dígitos de cada Id
df["Id"] = "P" + numeros.str.zfill(4)           # prefixo P + zeros à esquerda até 4 dígitos

# [V] Há Ids duplicados? E amostra do resultado
print("Ids duplicados:", int(df["Id"].duplicated().sum()))
print(df["Id"].head())

# [T-k] Limpar espaços do Nome (pontas e duplicados internos)
df["Nome"] = df["Nome"].str.strip().str.replace(r"\s+", " ", regex=True)

Ids duplicados: 0
1    P0002
3    P0004
5    P0006
6    P0007
7    P0008
Name: Id, dtype: str


In [26]:
# [R] Revisão geral da base limpa
df = df.reset_index(drop=True)      # renumera o índice 0..392 (tinha buracos dos descartes)
print("Formato final:", df.shape)
print()
df.info()
print()
print(df[["Glicemia_Inicial", "Glicemia_Final"]].describe())

Formato final: (393, 11)

<class 'pandas.DataFrame'>
RangeIndex: 393 entries, 0 to 392
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Id                393 non-null    str           
 1   Nome              393 non-null    str           
 2   Data_Nascimento   383 non-null    datetime64[us]
 3   Remedio_1         393 non-null    str           
 4   Remedio_2         393 non-null    str           
 5   Data_Inicio       393 non-null    datetime64[us]
 6   Data_Fim          393 non-null    datetime64[us]
 7   Glicemia_Inicial  393 non-null    float64       
 8   Glicemia_Final    393 non-null    float64       
 9   Dieta             393 non-null    str           
 10  Medicamento       393 non-null    str           
dtypes: datetime64[us](3), float64(2), str(6)
memory usage: 33.9 KB

       Glicemia_Inicial  Glicemia_Final
count        393.000000      393.000000
mean         164.586005      

In [27]:
# [T-l] Criar a variável-chave: redução de glicemia (inicial - final)
df["Reducao"] = df["Glicemia_Inicial"] - df["Glicemia_Final"]
print(df["Reducao"].describe())

# [S] Salvar a base limpa (checkpoint da Fase 1)
df.to_csv("glicemia_limpa.csv", index=False, encoding="utf-8")

count    393.000000
mean      32.235878
std       26.704321
min     -145.000000
25%       25.000000
50%       34.000000
75%       42.000000
max      169.000000
Name: Reducao, dtype: float64


In [28]:
# [T-m] Arredondar glicemia e redução para 1 casa (remove o ruído de ponto flutuante)
df["Glicemia_Inicial"] = df["Glicemia_Inicial"].round(1)
df["Glicemia_Final"]   = df["Glicemia_Final"].round(1)
df["Reducao"]          = df["Reducao"].round(1)

df.to_csv("glicemia_limpa.csv", index=False, encoding="utf-8")   # regrava, sempre UTF-8

# [V] Prova de que o arquivo está certo: recarregar no pandas
teste = pd.read_csv("glicemia_limpa.csv", encoding="utf-8")
print(teste[["Nome", "Dieta", "Reducao"]].head())

                      Nome Dieta  Reducao
0     Leandro Ramos Mendes   Sim     47.0
1     Carlos Dias Ferreira   Sim     26.5
2  Vinícius Teixeira Ramos   Não     21.0
3       Maria Almeida Lima   Não     24.0
4   Camila Pereira Freitas   Sim     46.0


# Fase 2 - Análise estatística

In [30]:
%pip install scipy

   ---------------------------------------- 0.0/37.4 MB ? eta -:--:--
   ------------------- -------------------- 18.6/37.4 MB 104.8 MB/s eta 0:00:01
   ---------------------------------------  37.2/37.4 MB 110.6 MB/s eta 0:00:01
   ---------------------------------------- 37.4/37.4 MB 72.8 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
# Fase 2 — Análise 1: o tratamento reduziu a glicemia? (teste t pareado)
from scipy import stats

t_stat, p_valor = stats.ttest_rel(df["Glicemia_Inicial"], df["Glicemia_Final"])

print("Estatística t:", round(t_stat, 3))
print("Valor-p:", p_valor)

Estatística t: 23.931
Valor-p: 1.1557547764942398e-78


In [32]:
# Análise 2: redução média por medicamento
print(df.groupby("Medicamento")["Reducao"].mean())

Medicamento
Remedio_1    36.778469
Remedio_2    27.076087
Name: Reducao, dtype: float64


In [33]:
# Análise 2: qual remédio reduz mais? (teste t independente)
grupo1 = df[df["Medicamento"] == "Remedio_1"]["Reducao"]
grupo2 = df[df["Medicamento"] == "Remedio_2"]["Reducao"]

t_stat, p_valor = stats.ttest_ind(grupo1, grupo2, equal_var=False)

print("Estatística t:", round(t_stat, 3))
print("Valor-p:", p_valor)

Estatística t: 3.61
Valor-p: 0.0003491674201660646


In [34]:
# Análise 3 — passo 1: dieta e medicamento estão embaralhados? (checar confusão)
print(pd.crosstab(df["Dieta"], df["Medicamento"], normalize="index"))
print()

# Análise 3 — passo 2: comparação ingênua da redução por dieta (teste t independente)
com_dieta = df[df["Dieta"] == "Sim"]["Reducao"]
sem_dieta = df[df["Dieta"] == "Não"]["Reducao"]
print("Média com dieta:", round(com_dieta.mean(), 2))
print("Média sem dieta:", round(sem_dieta.mean(), 2))
t_stat, p_valor = stats.ttest_ind(com_dieta, sem_dieta, equal_var=False)
print("t:", round(t_stat, 3), "| valor-p:", p_valor)

Medicamento  Remedio_1  Remedio_2
Dieta                            
Não           0.548718   0.451282
Sim           0.515152   0.484848

Média com dieta: 39.79
Média sem dieta: 24.57
t: 5.901 | valor-p: 8.548300383039225e-09


In [35]:
# Análise 4: intervalo de confiança de 95% para a vantagem da dieta
resultado = stats.ttest_ind(com_dieta, sem_dieta, equal_var=False)
ic = resultado.confidence_interval(confidence_level=0.95)

print("Vantagem estimada da dieta:", round(com_dieta.mean() - sem_dieta.mean(), 2))
print("IC 95%: de", round(ic.low, 2), "a", round(ic.high, 2))
print("Valor-p:", resultado.pvalue)

Vantagem estimada da dieta: 15.22
IC 95%: de 10.14 a 20.29
Valor-p: 8.548300383039225e-09
